# Branching and Merging

## What you'll learn

- Split a data stream into independent branches (fan-out)
- Produce multiple output variants from a single input
- Recombine branches with Merge (passthrough semantics)
- Build a complete branch-and-reconverge pipeline

**Prerequisites:** [Sources and Sequencing](01-sources-and-sequencing.ipynb)  
**Estimated time:** 10 minutes

---

Branching and variants let you compare transformations of the same data. Merge
combines their outputs for further processing.

| Pattern | What it does |
|---|---|
| **Branching** | Route one step's output to multiple downstream steps |
| **Variants** | One input produces multiple outputs within a single step |
| **Merge** | Combine artifact streams back into one |

In [ ]:
from __future__ import annotations

from artisan.operations.curator import Merge
from artisan.operations.examples import DataGenerator, DataTransformer
from artisan.orchestration import PipelineManager
from artisan.utils import tutorial_setup
from artisan.visualization import build_macro_graph, build_micro_graph

> **Graph legend:** See [Sources and Sequencing](01-sources-and-sequencing.ipynb) for the box/arrow key.

## Branching

Pass one step's output to multiple downstream steps. Each branch processes the
same input artifact IDs independently. Here the branches use different
transformation seeds.

In [ ]:
env_branch = tutorial_setup("branching")

pipeline = PipelineManager.create(
    name="branching",
    delta_root=env_branch.delta_root,
    staging_root=env_branch.staging_root,
    working_root=env_branch.working_root,
)
output = pipeline.output

pipeline.run(operation=DataGenerator, name="generate", params={"count": 2, "seed": 42})

# Both branches consume the same output — they use different params
pipeline.run(
    operation=DataTransformer,
    inputs={"dataset": output("generate", "datasets")},
    params={"seed": 100},
    name="transform_a",
)
pipeline.run(
    operation=DataTransformer,
    inputs={"dataset": output("generate", "datasets")},
    params={"seed": 200},
    name="transform_b",
)

result = pipeline.finalize()

In [ ]:
build_macro_graph(env_branch.delta_root)

Both transform steps receive the same two input artifacts and produce their
own outputs. Passing the same `OutputReference` to multiple `pipeline.run()`
calls creates the branch.

In [ ]:
build_micro_graph(env_branch.delta_root)

## Variants (in-step fan-out)

Branching splits the pipeline graph. Variants split *within* a single step:
one input artifact produces multiple output artifacts. The `variants`
parameter on `DataTransformer` controls this — each variant gets a
different random noise pattern.

| Branching | Variants |
|---|---|
| Multiple steps, same inputs | One step, multiple outputs per input |
| Different operations or params per branch | Same operation, automatic variation |
| Separate output streams | Single output stream (all variants together) |

In [ ]:
env_variants = tutorial_setup("variants")

pipeline = PipelineManager.create(
    name="variants",
    delta_root=env_variants.delta_root,
    staging_root=env_variants.staging_root,
    working_root=env_variants.working_root,
)
output = pipeline.output

pipeline.run(operation=DataGenerator, name="generate", params={"count": 2, "seed": 42})

# Each of the 2 input datasets produces 3 variants = 6 total outputs
pipeline.run(
    operation=DataTransformer,
    name="transform",
    inputs={"dataset": output("generate", "datasets")},
    params={"variants": 3, "seed": 100},
)

result = pipeline.finalize()

In [ ]:
build_macro_graph(env_variants.delta_root)

The transform step consumed 2 artifacts and produced 6 (3 variants
each). All outputs live in the same step and the same output role. The
provenance graph tracks which variant came from which input via lineage
edges (orange arrows).

In [ ]:
build_micro_graph(env_variants.delta_root)

## Merge

Merge returns the union of its input artifact IDs without creating new
artifacts. All streams must contain the same artifact type. The output role
is always `"merged"`.

In [ ]:
env_merge = tutorial_setup("merge")

pipeline = PipelineManager.create(
    name="merge",
    delta_root=env_merge.delta_root,
    staging_root=env_merge.staging_root,
    working_root=env_merge.working_root,
)
output = pipeline.output

pipeline.run(operation=DataGenerator, name="gen_a", params={"count": 2, "seed": 42})
pipeline.run(operation=DataGenerator, name="gen_b", params={"count": 3, "seed": 99})

# Merge accepts arbitrary role names — they're only labels
pipeline.run(
    operation=Merge,
    name="merge",
    inputs={
        "a": output("gen_a", "datasets"),
        "b": output("gen_b", "datasets"),
    },
)

result = pipeline.finalize()

In [ ]:
build_macro_graph(env_merge.delta_root)

`output("merge", "merged")` supplies all 5 existing artifacts: 2 from one
stream and 3 from the other.

In [ ]:
build_micro_graph(env_merge.delta_root)

Merge scales to any number of input streams. The input role names
(`"a"`, `"b"`, `"c"`, ...) are arbitrary labels — pick whatever is
meaningful for your pipeline.

## Putting it together: branch and reconverge

Combine branching and merge: generate
data, branch into independent transformations, merge the results back
together, then continue processing the combined stream.

```
                ┌─ transform_a ─┐
  generate ─────┤               ├─── merge ─── downstream
                └─ transform_b ─┘
```

In [ ]:
env_roundtrip = tutorial_setup("roundtrip")

pipeline = PipelineManager.create(
    name="roundtrip",
    delta_root=env_roundtrip.delta_root,
    staging_root=env_roundtrip.staging_root,
    working_root=env_roundtrip.working_root,
)
output = pipeline.output

# Generate source data
pipeline.run(operation=DataGenerator, name="generate", params={"count": 2, "seed": 42})

# Branch into two transformation strategies
pipeline.run(
    operation=DataTransformer,
    name="transform_a",
    inputs={"dataset": output("generate", "datasets")},
    params={"scale_factor": 0.5, "seed": 100},
)
pipeline.run(
    operation=DataTransformer,
    name="transform_b",
    inputs={"dataset": output("generate", "datasets")},
    params={"scale_factor": 2.0, "seed": 200},
)

# Merge the branches back together
pipeline.run(
    operation=Merge,
    name="merge",
    inputs={
        "small": output("transform_a", "dataset"),
        "large": output("transform_b", "dataset"),
    },
)

# Process the combined stream
pipeline.run(
    operation=DataTransformer,
    name="final_transform",
    inputs={"dataset": output("merge", "merged")},
    params={"noise_amplitude": 0.01, "seed": 300},
)

result = pipeline.finalize()

In [ ]:
build_macro_graph(env_roundtrip.delta_root)

The final step receives 4 artifacts, 2 from each branch. Provenance traces
its outputs back through the branches to the original inputs.

In [ ]:
build_micro_graph(env_roundtrip.delta_root)

## Next steps

- [Metrics and Filtering](03-metrics-and-filtering.ipynb) — score artifacts with metrics, then filter by threshold
- [Operations Model](../../concepts/operations-model.md) — How creators produce artifacts and curators route them
- [Execution Flow](../../concepts/execution-flow.md) — what happens when `pipeline.run()` executes a step